In [5]:
#### Thins the model training sample to correct for within-country spatial autocorrelation 
# Only applied to countries with many regions in training data
# Thinning method: repeatedly pick a random remaining region, drop every other remaining region
# whose polygon boundary is within the threshold distance of it (buffer zone), and repeat

from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

In [6]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import model data
DATASETS = {
    "capital": f"{cd}/Data/Clean/Training_data/capital_relative_final.csv",
    "labor":   f"{cd}/Data/Clean/Training_data/labor_relative_final.csv",
}

# Import polygon geometries (one row per PROJ_ID)
polygons = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")  
polygons = polygons[["PROJ_ID", "geometry"]]

# Set save path 
save_path = f"{cd}/Data/Clean/Training_data"

In [7]:
##### Set-up

THINNING_THRESHOLD_KM = 50   # empirically chosen from correlogram analysis
MIN_REGIONS_TO_THIN = 50      # countries with fewer regions than this are left untouched
RANDOM_SEED = 42              # change this to test sensitivity to which regions are kept

In [8]:
##### CORE FUNCTIONS

# picks an appropriate UTM CRS for a set of geometries based on their combined centroid, so within-country distances are measured accurately 
def get_utm_crs(gdf):
    centroid = gdf.geometry.union_all().centroid
    lon, lat = centroid.x, centroid.y
    utm_zone = int((lon + 180) / 6) + 1
    epsg = 32600 + utm_zone if lat >= 0 else 32700 + utm_zone
    return f"EPSG:{epsg}"

# function to spatially thin a single country's regions using true polygon-to-polygon distance
# repeatedly picks a random remaining region, drops all others within threshold_km of its
# boundary, and continues until every region has either been kept or dropped
def thin_country(sub_gdf, threshold_km, rng):
    utm_crs = get_utm_crs(sub_gdf)
    proj = sub_gdf.to_crs(utm_crs)
    threshold_m = threshold_km * 1000.0

    remaining = proj.copy()
    kept_idx = []

    while len(remaining) > 0:
        pick_idx = rng.choice(remaining.index.to_numpy())
        picked_geom = remaining.loc[pick_idx, "geometry"]
        kept_idx.append(pick_idx)

        dists = remaining.geometry.distance(picked_geom)
        remaining = remaining.loc[dists >= threshold_m]

    return sub_gdf.loc[kept_idx].drop(columns="geometry")

# function to thin an entire dataset: only thins countries with above the threshold regions
def thin_dataset(gdf, threshold_km, min_regions, rng):
    thinned_parts = []
    summary_rows = []

    for country, sub in gdf.groupby("country_ID"):
        n_before = len(sub)

        if n_before < min_regions:
            thinned_parts.append(sub.drop(columns="geometry"))
            summary_rows.append({
                "country_ID": country, "n_before": n_before,
                "n_after": n_before, "n_dropped": 0, "thinned": False,
            })
            continue

        thinned_sub = thin_country(sub, threshold_km, rng)
        n_after = len(thinned_sub)

        thinned_parts.append(thinned_sub)
        summary_rows.append({
            "country_ID": country, "n_before": n_before,
            "n_after": n_after, "n_dropped": n_before - n_after, "thinned": True,
        })

    thinned_df = pd.concat(thinned_parts, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows).sort_values("n_dropped", ascending=False)
    return thinned_df, summary_df

In [ ]:
##### Run

rng = np.random.default_rng(RANDOM_SEED)

for name, path in DATASETS.items():
    print(f"\n── {name} ──────────────────────────────")

    df = pd.read_csv(path)
    n_before_total = len(df)

    gdf = polygons.merge(df, on="PROJ_ID", how="right")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=polygons.crs)

    missing_geom = gdf["geometry"].isna().sum()
    if missing_geom > 0:
        print(f"  warning: {missing_geom} rows missing geometry, dropping before thinning")
        gdf = gdf.dropna(subset=["geometry"])

    thinned_df, summary_df = thin_dataset(gdf, THINNING_THRESHOLD_KM, MIN_REGIONS_TO_THIN, rng)

    print(summary_df.to_string(index=False))
    print(f"  total: {n_before_total:,} -> {len(thinned_df):,} rows "
          f"({n_before_total - len(thinned_df):,} dropped)")

    out_path = f"{save_path}/{name}_relative_final_thinned.csv"
    thinned_df.to_csv(out_path, index=False)
    print(f"  saved to {out_path}")


── capital ──────────────────────────────
